In [ ]:
## Google Drive root path and project path definitions
DRIVE_ROOT_PATH = '/content/drive/MyDrive'
RLCCSAM_ROOT_PATH = f"{DRIVE_ROOT_PATH}/RL-CC-SAM"

In [ ]:
## Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
## Ref: https://netraneupane.medium.com/how-to-install-libraries-permanently-in-google-colab-fb15a585d8a5

In [ ]:
## Create virtual env for this project - 1. Install virtualenv
!pip install virtualenv

In [ ]:
## Create virtual env for this project - 1. Create virtual env `llms`
!virtualenv {DRIVE_ROOT_PATH}/colab_envs/llms

In [ ]:
## Install Python libraries needed
!chmod +x {DRIVE_ROOT_PATH}/colab_envs/llms/bin/*
##!source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && pip install -r {RLCCSAM_ROOT_PATH}/requirements.txt
## Or
!source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && python -m pip install -r {RLCCSAM_ROOT_PATH}/requirements.txt

In [ ]:
## Pull submodules
!cd {RLCCSAM_ROOT_PATH} && git submodule update --init --recursive

In [ ]:
## Download pretrained models needed
!mkdir -p {RLCCSAM_ROOT_PATH}/pretrained
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O /content/drive/MyDrive/RL-CC-SAM/pretrained/sam_vit_h_4b8939.pth

In [ ]:
## Fix AWS CLI installation - force reinstall to create executable
!source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && pip install --force-reinstall --no-cache-dir awscli
!echo "Checking AWS CLI installation:"
!ls -la {DRIVE_ROOT_PATH}/colab_envs/llms/bin/aws* || echo "AWS executable still missing, will use Python module"


In [ ]:
## Download MedSAM pretrained model
!mkdir -p {RLCCSAM_ROOT_PATH}/pretrained
!wget -O {RLCCSAM_ROOT_PATH}/pretrained/medsam_vit_b.pth "https://zenodo.org/records/10689643/files/medsam_vit_b.pth" || echo "MedSAM download may have failed, will try alternative source"
# Alternative source if primary fails
![ ! -f {RLCCSAM_ROOT_PATH}/pretrained/medsam_vit_b.pth ] && wget -O {RLCCSAM_ROOT_PATH}/pretrained/medsam_vit_b.pth "https://huggingface.co/flaviagiammarino/medsam-vit-base/resolve/main/pytorch_model.bin" || echo "MedSAM model download completed"

In [ ]:
!cd {RLCCSAM_ROOT_PATH}/datasets && wget -O FLARE22Train.zip "https://zenodo.org/records/7860267/files/FLARE22Train.zip?download=1" || echo "FLARE22Train download may have failed"
!cd {RLCCSAM_ROOT_PATH}/datasets && [ -f FLARE22Train.zip ] && unzip -q FLARE22Train.zip -d .

In [ ]:
## List downloaded files for verification
!echo "=== Downloaded Pretrained Models ==="
!ls -lh {RLCCSAM_ROOT_PATH}/pretrained/

In [ ]:

## Download Medical Decathlon datasets for MedSAM baseline testing

## Create datasets directory structure
!mkdir -p {RLCCSAM_ROOT_PATH}/datasets/medical_decathlon
!mkdir -p {RLCCSAM_ROOT_PATH}/datasets/medical_sam_results

# Task04 - Hippocampus (MRI) dataset (~29MB)
!cd {RLCCSAM_ROOT_PATH}/datasets/medical_decathlon && source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && aws s3 cp --no-sign-request s3://msd-for-monai/Task04_Hippocampus.tar . || echo "Hippocampus download may have failed"
!cd {RLCCSAM_ROOT_PATH}/datasets/medical_decathlon && [ -f Task04_Hippocampus.tar ] && tar -xf Task04_Hippocampus.tar && rm Task04_Hippocampus.tar || echo "Hippocampus extraction completed or skipped"

# Task09 - Spleen (CT) dataset (~1.5GB)
!cd {RLCCSAM_ROOT_PATH}/datasets/medical_decathlon && source {DRIVE_ROOT_PATH}/colab_envs/llms/bin/activate && aws s3 cp --no-sign-request s3://msd-for-monai/Task09_Spleen.tar . || echo "Spleen download may have failed"
!cd {RLCCSAM_ROOT_PATH}/datasets/medical_decathlon && [ -f Task09_Spleen.tar ] && tar -xf Task09_Spleen.tar && rm Task09_Spleen.tar || echo "Spleen extraction completed or skipped"

In [ ]:
# BUSI - Breast Ultrasound dataset (~250MB)
!cd {RLCCSAM_ROOT_PATH}/datasets && wget -O BUSI.zip "https://data.mendeley.com/public-files/datasets/wmy84r64cy/files/1b1b6b6c-5d3f-4e96-b0a2-42bb13e7ea32/file_downloaded" || echo "BUSI download may have failed"
!cd {RLCCSAM_ROOT_PATH}/datasets && [ -f BUSI.zip ] && unzip -q BUSI.zip -d BUSI_Dataset && rm BUSI.zip || echo "BUSI extraction completed or skipped"

In [ ]:
!echo "=== Downloaded Medical Datasets ==="
!ls -lh {RLCCSAM_ROOT_PATH}/datasets/